# Overall synchrony changes (zero lag)


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from notebook_setup import init
FINAL_DIR, PDF_DIR = init()
import plot_style

from analysis_config import BASE_PATH, FOCUS_AUS, ALPHA
from dyad_analysis_utils import (
    load_openface_data,
    load_prepared_meetings,
    organize_sessions_by_period,
    compute_meeting_fisher_matrices,
    compute_mean_abs_fisher,
    run_rm_anova_three_periods,
    run_posthoc_period_tests,
)

alpha = ALPHA
base_path = BASE_PATH
if not base_path.exists():
    raise FileNotFoundError(f"Expected directory not found: {base_path.resolve()}")

meeting_groups, input_paths = load_openface_data(base_path)
print(f"Found {len(input_paths)} session folders, {len(meeting_groups)} meetings")

In [ ]:
df_infant_arr, df_adult_arr, AUs = load_prepared_meetings(meeting_groups)
print(f"Loaded {len(df_infant_arr)} meetings")


In [ ]:
period_aggregated_data, all_periods_data = organize_sessions_by_period(
    df_infant_arr, df_adult_arr, AUs
)
for p in (1, 2, 3):
    print(f"Period {p}: {len(period_aggregated_data[p])} sessions")


In [ ]:
print(f"Using FOCUS_AUS ({len(FOCUS_AUS)}): {FOCUS_AUS}")


In [ ]:
meeting_fisher_matrices = {
    meeting_id: compute_meeting_fisher_matrices(
        df_infant_arr[meeting_id], df_adult_arr[meeting_id], FOCUS_AUS
    )
    for meeting_id in df_infant_arr
}
print(f"Computed Fisher Z matrices for {len(meeting_fisher_matrices)} meetings")


In [ ]:
summary_table = compute_mean_abs_fisher(meeting_fisher_matrices)
summary_table_complete = summary_table.dropna(subset=["period_1", "period_2", "period_3"])

print(f"Meetings with all 3 periods: {len(summary_table_complete)}")
for period in (1, 2, 3):
    col = summary_table_complete[f"period_{period}"]
    print(f"{plot_style.month_label(period)}: mean |Fisher z| = {col.mean():.3f} (SD = {col.std():.3f})")


In [ ]:
period_1_vals = summary_table_complete['period_1'].values
period_2_vals = summary_table_complete['period_2'].values
period_3_vals = summary_table_complete['period_3'].values

print("=" * 60)
print("NORMALITY OF PAIRWISE DIFFERENCES (SHAPIRO-WILK)")
print("=" * 60)

rm_result = run_rm_anova_three_periods(period_1_vals, period_2_vals, period_3_vals, alpha=alpha)
display(rm_result['normality'])
print(f"All differences normal (p >= {alpha}): {rm_result['is_normal']}")

print("\n" + "=" * 60)
print("REPEATED-MEASURES ANOVA")
print("=" * 60)
print(f"Greenhouse-Geisser epsilon: {rm_result['eps']:.4f}")
if rm_result['eps'] >= 0.75:
    print("Sphericity adequate (eps >= 0.75); using uncorrected p-value.")
else:
    print("Sphericity violated (eps < 0.75); using Greenhouse-Geisser corrected p-value.")
print(f"p (uncorrected): {rm_result['p_unc']:.6f}")
print(f"p (GG):          {rm_result['p_gg']:.6f}")
display(rm_result['anova_table'])

test_name = rm_result['test_name']
main_test_stat = rm_result['statistic']
main_test_p = rm_result['p_value']
print(f"\n{test_name}: F = {main_test_stat:.2f}, p = {main_test_p:.4f} (significant: {main_test_p < alpha})")


In [ ]:
print("=" * 60)
print("POST-HOC ANALYSIS")
print("=" * 60)

if main_test_p < alpha:
    print(f"Main test was significant (p = {main_test_p:.4f}).")
    print("Running post-hoc paired comparisons with Holm-Bonferroni correction...\n")
    posthoc_df = run_posthoc_period_tests(period_1_vals, period_2_vals, period_3_vals, alpha=alpha)
    print("Post-hoc results:")
    display(posthoc_df)

    print("\n" + "-" * 40)
    print("SUMMARY")
    print("-" * 40)
    n_sig = posthoc_df['Significant (Holm)'].sum()
    print(f"Significant comparisons after Holm correction: {n_sig}/3")

    if n_sig > 0:
        print("\nSignificant differences:")
        for _, row in posthoc_df[posthoc_df['Significant (Holm)']].iterrows():
            print(f"  - {row['Comparison']}: p = {row['Corrected p-value (Holm)']:.4f}")
else:
    print(f"Main test was NOT significant (p = {main_test_p:.4f}).")
    print("No post-hoc tests needed.")
    posthoc_df = None
